# Regularization and Training Control

Goal: understand techniques that improve neural-network generalization and training stability.

Topics:
- dropout
- batch normalization
- learning-rate scheduling
- comparing regularized models

In [1]:
# tools imported

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader, random_split
import matplotlib.pyplot as plt

In [2]:
# noisy nonlinear dataset creation

torch.manual_seed(42)

x = torch.linspace(-3, 3, 120).reshape(-1, 1)

y = (
    0.5 * x**3
    - 2 * x**2
    + x
    + 3
)

y += 1.5 * torch.randn_like(y)

In [3]:
# training / validation split

dataset = TensorDataset(x, y)

train_dataset, val_dataset = random_split(
    dataset,
    [60, 60],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=10,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=10,
    shuffle=False
)

In [4]:
# set device

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using:", device)

Using: cuda


## Dropout Method

Dropout randomly disables some neuron outputs during training, with the idea of stopping the network from depending too much on certain neurons

In [6]:
# model definition

class DropoutNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(1, 256), # linear
            nn.ReLU(), # ReLU
            nn.Dropout(p=0.2), # dropout

            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Dropout(p=0.2),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(p=0.2),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.network(x)

In [7]:
dropout_model = DropoutNetwork().to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    dropout_model.parameters(),
    lr=0.001
)

In [8]:
# training loop

epochs = 1500

train_losses = []
val_losses = []

for epoch in range(epochs):

    dropout_model.train()
    train_loss = 0.0

    for xb, yb in train_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        predictions = dropout_model(xb)

        loss = criterion(predictions, yb)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    dropout_model.eval()
    val_loss = 0.0

    with torch.no_grad():

        for xb, yb in val_loader:

            xb = xb.to(device)
            yb = yb.to(device)

            predictions = dropout_model(xb)

            loss = criterion(predictions, yb)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if epoch % 100 == 0:
        print(
            f"Epoch {epoch:4d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f}"
        )
    

Epoch    0 | Train Loss: 91.8510 | Val Loss: 75.1506
Epoch  100 | Train Loss: 3.7014 | Val Loss: 3.2428
Epoch  200 | Train Loss: 2.5904 | Val Loss: 2.4658
Epoch  300 | Train Loss: 3.7577 | Val Loss: 4.7457
Epoch  400 | Train Loss: 2.1106 | Val Loss: 2.5339
Epoch  500 | Train Loss: 1.8473 | Val Loss: 2.9888
Epoch  600 | Train Loss: 3.1956 | Val Loss: 6.2014
Epoch  700 | Train Loss: 1.7848 | Val Loss: 2.8322
Epoch  800 | Train Loss: 2.9504 | Val Loss: 3.4732
Epoch  900 | Train Loss: 2.2494 | Val Loss: 3.7141
Epoch 1000 | Train Loss: 3.0991 | Val Loss: 3.5903
Epoch 1100 | Train Loss: 2.7519 | Val Loss: 2.7085
Epoch 1200 | Train Loss: 1.8737 | Val Loss: 3.3178
Epoch 1300 | Train Loss: 1.7511 | Val Loss: 2.9676
Epoch 1400 | Train Loss: 2.7381 | Val Loss: 2.6975
